# Baseline DQN

Thin runner for the Leurent/rl-agents-style DQN baseline.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
helper_dir_str = str(HELPER_DIR)
if helper_dir_str not in sys.path:
    sys.path.insert(0, helper_dir_str)

from dqn_notebook_utils import (
    build_dqn_args,
    build_env_config,
    evaluate_saved_model,
    load_dqn_backend,
    show_policy_panel,
    train_and_display,
)

trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
    backend_module="elurant_dqn",
    notebook_subdir="baseline_dqn",
    results_subdir="baseline_dqn",
)

## Config

In [ ]:
if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
    from pathlib import Path
    import sys

    def find_project_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().parents]:
            if (candidate / "src").exists() and (candidate / ".git").exists():
                return candidate
        raise RuntimeError("Could not locate the project root.")

    PROJECT_ROOT = find_project_root()
    HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
    helper_dir_str = str(HELPER_DIR)
    if helper_dir_str not in sys.path:
        sys.path.insert(0, helper_dir_str)

    from dqn_notebook_utils import build_dqn_args, build_env_config, load_dqn_backend

    trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
        backend_module="elurant_dqn",
        notebook_subdir="baseline_dqn",
        results_subdir="baseline_dqn",
    )

environment_profile = "structured_baseline"
eval_environment_profile = environment_profile

environment_overrides = {
    "lanes_count": 3,
    "vehicles_count": 20,
    "duration": 40,
    "ego_spacing": 2.0,
    "vehicles_density": 1.0,
    "simulation_frequency": 15,
    "policy_frequency": 1,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
}
# Evaluation environment: choose a profile independently from training.
# Leave overrides empty unless you intentionally want to override that
# eval profile's defaults.
eval_environment_overrides = {}

observation_config = {
    "vehicles_count": 5,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
}
action_config = {
    "type": "DiscreteMetaAction",
}
reward_config = {
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "lane_change_reward": 0.0,
    "normalize_reward": True,
}
min_reward_speed = 20.0
max_reward_speed = 30.0
speed_config = {
    "reward_speed_range": [min_reward_speed, max_reward_speed],
}
target_speed_range = {
    "min_target_speed": 10.0,
    "max_target_speed": 35.0,
}
adaptive_longitudinal_config = {
    "enabled": False,
    "ttc_midpoint": 4.0,
    "ttc_temperature": 1.0,
    "ttc_cap": 10.0,
    **target_speed_range,
    "faster_max_delta": 1.25,
    "slower_min_delta": 1.25,
    "slower_max_delta": 2.5,
}

timesteps = 20_000
n_envs = 4
eval_episodes = 5
seed = 42
run_name = "baseline_dqn_tuned_20k"
train_freq = 4
gradient_steps = train_freq * n_envs

hyperparameters = {
    "learning_rate": 2.5e-4,
    "buffer_size": 50_000,
    "learning_starts": 2_000,
    "batch_size": 64,
    "gamma": 0.95,
    "target_update_interval": 1_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.70,
    "exploration_final_eps": 0.10,
    "progress_every": 1_000,
    "verbose": 0,
}


## Driver Behavior Spectrum

This opt-in baseline DQN config assigns each surrounding IDM/MOBIL vehicle a continuous aggressiveness score from 0 to 100. Blue vehicles are conservative; red vehicles are aggressive.


In [ ]:
import matplotlib.pyplot as plt
import importlib
import dqn_notebook_utils as dqn_utils

dqn_utils = importlib.reload(dqn_utils)
build_dqn_args = dqn_utils.build_dqn_args
build_env_config = dqn_utils.build_env_config
evaluate_saved_model = dqn_utils.evaluate_saved_model
show_policy_panel = dqn_utils.show_policy_panel
train_and_display = dqn_utils.train_and_display
trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = dqn_utils.load_dqn_backend(
    backend_module="elurant_dqn",
    notebook_subdir="baseline_dqn",
    results_subdir="baseline_dqn",
)


# Continuous driver behavior spectrum for baseline DQN traffic only.
# score=0 is conservative/blue; score=100 is aggressive/red.
driver_aggressiveness_config = {
    "enabled": True,
    "distribution": "uniform",  # use "normal" or set fixed_score for controlled sweeps
    "min_score": 0.0,
    "max_score": 100.0,
    "fixed_score": None,
    "normal_mean": 50.0,
    "normal_std": 20.0,
    "conservative": {
        "target_speed": 20.0,
        "acc_max": 4.0,
        "comfort_acc_max": 2.0,
        "comfort_acc_min": -4.0,
        "delta": 4.5,
        "time_wanted": 2.4,
        "distance_wanted": 14.0,
        "politeness": 0.8,
        "lane_change_min_acc_gain": 0.8,
        "lane_change_max_braking_imposed": 1.0,
        "lane_change_delay": 1.5,
    },
    "aggressive": {
        "target_speed": 30.0,
        "acc_max": 7.0,
        "comfort_acc_max": 5.5,
        "comfort_acc_min": -6.5,
        "delta": 3.5,
        "time_wanted": 0.6,
        "distance_wanted": 6.0,
        "politeness": 0.0,
        "lane_change_min_acc_gain": 0.05,
        "lane_change_max_braking_imposed": 3.5,
        "lane_change_delay": 0.5,
    },
}

run_name = "baseline_dqn_driver_spectrum_20k"
training_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=environment_overrides,
    observation=observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    adaptive_longitudinal=adaptive_longitudinal_config,
    driver_aggressiveness=driver_aggressiveness_config,
)
saved_model_eval_env_config = build_env_config(
    profile_name=eval_environment_profile,
    profile_overrides=eval_environment_overrides,
    observation=observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    adaptive_longitudinal=adaptive_longitudinal_config,
    driver_aggressiveness=driver_aggressiveness_config,
)
args = build_dqn_args(
    results_dir=RESULTS_DIR,
    run_name=run_name,
    timesteps=timesteps,
    eval_episodes=eval_episodes,
    seed=seed,
    num_envs=n_envs,
    device=DEFAULT_DEVICE,
    hyperparameters=hyperparameters,
)

saved_model_eval_episodes = 1000
saved_model_eval_seed = seed + 10000
saved_model_eval_name = f"saved_model_eval_{saved_model_eval_episodes}_episodes"
visualization_episodes = 5
visualization_max_steps = 300
visualization_seed = seed + 20000
visualization_stochastic = False

preview_env = trainer.make_env(render_mode="rgb_array", config=training_env_config)
try:
    _, preview_info = preview_env.reset(seed=seed)
    controlled_ids = {id(vehicle) for vehicle in getattr(preview_env.unwrapped, "controlled_vehicles", [])}
    preview_rows = []
    for idx, vehicle in enumerate(getattr(preview_env.unwrapped.road, "vehicles", [])):
        if id(vehicle) in controlled_ids or not hasattr(vehicle, "driver_aggressiveness_score"):
            continue
        preview_rows.append(
            {
                "vehicle": len(preview_rows),
                "score": round(float(vehicle.driver_aggressiveness_score), 1),
                "color_rgb": tuple(int(channel) for channel in vehicle.color),
                "target_speed": round(float(vehicle.target_speed), 1),
                "time_wanted": round(float(vehicle.TIME_WANTED), 2),
                "politeness": round(float(vehicle.POLITENESS), 2),
                "lane_gain": round(float(vehicle.LANE_CHANGE_MIN_ACC_GAIN), 2),
            }
        )
        if len(preview_rows) >= 10:
            break

    display(pd.DataFrame(preview_rows))
    display(pd.DataFrame({"training": pd.Series(training_env_config), "saved_eval": pd.Series(saved_model_eval_env_config)}))

    frame = preview_env.render()
    plt.figure(figsize=(10, 3))
    plt.imshow(frame)
    plt.axis("off")
    plt.show()
finally:
    preview_env.close()


## Train

In [ ]:
summary = train_and_display(
    trainer,
    args,
    training_env_config,
    label="Baseline DQN",
)

## Saved-Model Evaluation

In [ ]:
saved_eval_summary = evaluate_saved_model(
    trainer,
    summary_path=RESULTS_DIR / run_name / "summary.json",
    env_config=saved_model_eval_env_config,
    episodes=saved_model_eval_episodes,
    seed=saved_model_eval_seed,
    name=saved_model_eval_name,
    label="Baseline DQN",
)

## Congestion Failure Diagnostics

Label each evaluation episode with congestion failure modes: bad action, no good discrete action, wrong lane earlier, and unavoidable rear pressure. The per-step traces are saved so individual failures can be inspected afterward.


In [ ]:
import json
from pathlib import Path

from stable_baselines3 import DQN

from congestion_diagnostics import evaluate_congestion_diagnostics, save_congestion_diagnostics

congestion_diagnostic_episodes = 100
congestion_diagnostic_seed = seed + 50_000
congestion_diagnostic_config = {
    "ttc_cap": 10.0,
    "front_ttc_safe": 4.0,
    "front_ttc_critical": 1.5,
    "rear_ttc_safe": 4.0,
    "rear_ttc_critical": 1.5,
    "lane_gap_safe": 12.0,
    "bad_action_margin": 0.35,
    "no_good_action_risk": 0.85,
    "wrong_lane_quality_margin": 0.18,
    "wrong_lane_lookback_steps": 6,
    "final_lookback_steps": 4,
}

congestion_summary_path = RESULTS_DIR / run_name / "summary.json"
if not congestion_summary_path.exists():
    raise RuntimeError("Run the baseline training cell once so a saved model exists.")

congestion_saved_summary = json.loads(congestion_summary_path.read_text(encoding="utf-8"))
congestion_model = DQN.load(congestion_saved_summary["model_path"], device=DEFAULT_DEVICE)
congestion_output_dir = RESULTS_DIR / run_name / f"congestion_diagnostics_{congestion_diagnostic_episodes}_episodes"

congestion_df, congestion_traces = evaluate_congestion_diagnostics(
    congestion_model,
    trainer.make_env,
    env_config=saved_model_eval_env_config,
    episodes=congestion_diagnostic_episodes,
    seed=congestion_diagnostic_seed,
    diagnostic_config=congestion_diagnostic_config,
)
congestion_paths = save_congestion_diagnostics(congestion_df, congestion_traces, congestion_output_dir)

label_columns = [
    "collision",
    "agent_chose_badly",
    "no_good_discrete_action",
    "wrong_lane_earlier",
    "unavoidable_rear_pressure",
]
label_rates = (
    congestion_df[label_columns]
    .astype(float)
    .mean()
    .mul(100.0)
    .rename("rate_percent")
    .reset_index()
    .rename(columns={"index": "label"})
)
collision_breakdown = (
    congestion_df["collision_type"]
    .value_counts(dropna=False)
    .rename_axis("collision_type")
    .reset_index(name="episodes")
)

print(json.dumps(congestion_paths, indent=2))
display(label_rates.round(2))
display(collision_breakdown)
display(congestion_df.head(20))


## Safety TTC + Flow Reward Baseline


In [ ]:
# Retrain baseline DQN with a safety TTC + anti-lag reward term, without rear-flow injection.
safety_observation_config = {
    "vehicles_count": 10,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
    "see_behind": True,
}
safety_rear_flow_config = {
    "enabled": False,
}
safety_ttc_flow_reward_config = {
    "enabled": True,
    "ttc_safe_threshold": 4.0,
    "ttc_target": 6.0,
    "ttc_cap": 10.0,
    "low_ttc_penalty_weight": 0.7,
    "max_low_ttc_penalty": 0.9,
    "safe_ttc_bonus_weight": 0.08,
    "max_safe_ttc_bonus": 0.12,
    "lag_penalty_weight": 0.16,
    "speed_tolerance": 2.0,
    "max_lag_penalty": 0.9,
    "rear_ttc_pressure": 5.0,
    "rear_pressure_floor": 0.25,
    "flow_radius": 120.0,
    "lanes": "ego_and_adjacent",
}

safety_run_name = "baseline_dqn_safety_ttc_flow_20k"
safety_training_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=environment_overrides,
    observation=safety_observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    adaptive_longitudinal=adaptive_longitudinal_config,
    rear_flow=safety_rear_flow_config,
    traffic_flow_reward={"enabled": False},
    safety_ttc_flow_reward=safety_ttc_flow_reward_config,
)
safety_saved_model_eval_env_config = build_env_config(
    profile_name=eval_environment_profile,
    profile_overrides=eval_environment_overrides,
    observation=safety_observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    adaptive_longitudinal=adaptive_longitudinal_config,
    rear_flow=safety_rear_flow_config,
    traffic_flow_reward={"enabled": False},
    safety_ttc_flow_reward=safety_ttc_flow_reward_config,
)
safety_args = build_dqn_args(
    results_dir=RESULTS_DIR,
    run_name=safety_run_name,
    timesteps=timesteps,
    eval_episodes=eval_episodes,
    seed=seed + 300,
    num_envs=n_envs,
    device=DEFAULT_DEVICE,
    hyperparameters=hyperparameters,
)

display(pd.DataFrame({
    "training": pd.Series(safety_training_env_config),
    "saved_eval": pd.Series(safety_saved_model_eval_env_config),
}))

safety_summary = train_and_display(
    trainer,
    safety_args,
    safety_training_env_config,
    label="Baseline DQN + Safety TTC/Flow Reward",
)
safety_saved_eval_summary = evaluate_saved_model(
    trainer,
    summary_path=RESULTS_DIR / safety_run_name / "summary.json",
    env_config=safety_saved_model_eval_env_config,
    episodes=saved_model_eval_episodes,
    seed=saved_model_eval_seed + 300,
    name=saved_model_eval_name,
    label="Baseline DQN + Safety TTC/Flow Reward",
)


## Policy Panel

In [ ]:
policy_panel_rows = show_policy_panel(
    trainer,
    summary_path=RESULTS_DIR / run_name / "summary.json",
    env_config=saved_model_eval_env_config,
    episodes=visualization_episodes,
    max_steps=visualization_max_steps,
    seed=visualization_seed,
    stochastic=visualization_stochastic,
)